<a href="https://www.kaggle.com/code/avikdas567/diabetes-prediction-challenge?scriptVersionId=289427601" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e12/sample_submission.csv
/kaggle/input/playground-series-s5e12/train.csv
/kaggle/input/playground-series-s5e12/test.csv


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. Load data
train = pd.read_csv('/kaggle/input/playground-series-s5e12/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s5e12/test.csv')
sample_sub = pd.read_csv('/kaggle/input/playground-series-s5e12/sample_submission.csv')

# 2. Separate features and target
X = train.drop(columns=['id', 'diagnosed_diabetes'])
y = train['diagnosed_diabetes']
X_test = test.drop(columns=['id'])

# 3. Identify categorical columns
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns detected:", cat_cols)

# 4. Label encode categorical columns
le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col])
    X_test[col] = le.transform(X_test[col])

# 5. Scale only numeric (non-categorical) columns
num_cols = [col for col in X.columns if col not in cat_cols]
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# 6. LightGBM parameters
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 64,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 0.2,
    'lambda_l2': 0.2,
    'min_data_in_leaf': 30,
    'random_state': 42,
    'verbosity': -1
}

# 7. Cross validation training
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Fold {fold+1} ---")
    
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols)
    dvalid = lgb.Dataset(X_valid, label=y_valid, categorical_feature=cat_cols)

    model = lgb.train(
        params,
        dtrain,
        valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(200), lgb.log_evaluation(100)]
    )

    oof_preds[valid_idx] = model.predict(X_valid)
    test_preds += model.predict(X_test) / n_splits

    print(f"Fold {fold+1} AUC:", roc_auc_score(y_valid, oof_preds[valid_idx]))

# 8. Overall score
print("\nOverall CV AUC:", roc_auc_score(y, oof_preds))

# 9. Create submission
submission = pd.DataFrame({
    'id': test['id'],
    'diagnosed_diabetes': test_preds
})

submission.to_csv('submission.csv', index=False)
print("\nSubmission file created!")
print(submission.head())

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


Categorical columns detected: ['gender', 'ethnicity', 'education_level', 'income_level', 'smoking_status', 'employment_status']

--- Fold 1 ---
Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.717999
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.717999
Fold 1 AUC: 0.7179991691463289

--- Fold 2 ---
Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.71582
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.71582
Fold 2 AUC: 0.7158195913336864

--- Fold 3 ---
Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.716638
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.716638
Fold 3 AUC: 0.7166378332722398

--- Fold 4 ---
Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.717096
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.717096
Fold 4 AUC: 0.7170956391181016

--- F